# Chapter 18: Security (Reference)

## Learning Objectives

- Detect direct interpolation of untrusted context fields in a run: block
- See literally what GitHub would substitute for a malicious PR title
- Apply the env: indirection fix and confirm the detector clears it
- State why quoting alone does not prevent script injection

## Setup

The next cell sets up reproducibility and the `PRA_MODE` toggle. You should see `PRA_MODE = 'fixture'` printed by default. Note: this notebook never executes shell commands -- it only builds and inspects strings, so the "injection" is always just text you're reading, never anything run.

In [ ]:
import os
import random
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "pr_automerge").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

RANDOM_STATE: int = 42
random.seed(RANDOM_STATE)

PRA_MODE = os.environ.get("PRA_MODE", "fixture")
PRA_REPO = os.environ.get("PRA_REPO", "")

if PRA_MODE == "live":
    assert PRA_REPO, "Set PRA_REPO=owner/name to run against a real repo"

print(f"PRA_MODE = {PRA_MODE!r}")


## 1. Detecting the Unsafe Pattern

The next cell scans an unsafe `run:` block for direct interpolation of untrusted fields. You should see `github.event.pull_request.title` flagged.

In [ ]:
from labs.lab_18_security import find_direct_interpolation

unsafe_run = 'run: echo "Processing: ${{ github.event.pull_request.title }}"'
found = find_direct_interpolation(unsafe_run)
print(f"run block: {unsafe_run}")
print(f"untrusted fields interpolated directly: {found}")


## 2. The Literal Substitution

The next cell shows exactly what GitHub would substitute for a malicious PR title -- never executed, just inspected as a string. You should see the injected `curl | bash` become visible after the closing quote.

In [ ]:
from labs.lab_18_security import simulate_unsafe_substitution, MALICIOUS_TITLE

substituted = simulate_unsafe_substitution(unsafe_run, MALICIOUS_TITLE)
print(f"attacker-controlled title: {MALICIOUS_TITLE!r}")
print(f"substituted command: {substituted}")


## 3. The Fix

The next cell shows the `env:`-indirection fix and confirms the detector finds nothing unsafe in it. You should see an empty list.

In [ ]:
from labs.lab_18_security import safe_env_indirection

print(safe_env_indirection(unsafe_run))

safe_run_only = 'run: |\n  echo "Processing: $PR_TITLE"'
print(f"\nuntrusted fields found in the fixed run: block: {find_direct_interpolation(safe_run_only)}")


## Takeaways & Next Steps

This notebook's takeaway is Section 2's substituted command -- re-read it slowly and confirm you can see exactly where the shell's parsing would break.

In [ ]:
print("This repo's own five real workflows were audited with this exact detector -- zero hits.")


---

📖 **Reading companion:** [Chapter 18: Security](../learning_modules/chapter_18_security.md)
